# Build the merged Page-1/Page-2 dam dataset

**Goal.** Re-extract the complete accepted Page-1 and Page-2 PDF corpora, preserve all evidence, perform a one-to-one outer merge, and publish a fresh SQLite database, CSV products, investigation tables, figures, and execution receipt.

**Limitations.** The page extractors are frozen accepted science. The wrapper never averages, interpolates, zero-fills, or treats checksum totals as dam measurements. Page 3 and forecasting are out of scope.

## Architecture

```mermaid
flowchart LR
  A[Page-1 PDFs] --> B[Frozen Page-1 extractor]
  C[Page-2 PDFs] --> D[Frozen Page-2 extractor]
  B --> E[Page-1 observations + lineage + checks]
  D --> F[Page-2 observations + lineage + checks]
  E --> G[One-to-one outer merge]
  F --> G
  G --> H[Fresh SQLite]
  G --> I[Plain + compressed CSV]
  E --> J[Investigation]
  F --> J
  G --> J
  J --> K[Tables + 12 figures]
```

Data lineage: **source PDF → extracted observation → value lineage → page table → merged observation → SQLite → CSV**.

## Page semantics

Page 1 reports operational daily/cumulative inflow, release, and storage fields. Page 2 reports structural and corresponding operational fields. Similar-looking cross-page values are comparisons—not equivalences—and both values are retained with page prefixes. Observation identity is canonical dam identifier plus bulletin date. Page‑2 R.S. values and nonnumeric source expressions are governed protected source states, not extraction problems.

In [ ]:
from pathlib import Path
import sys, json, hashlib, sqlite3
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "DAM_DATA_PIPELINE_20260831").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "DAM_DATA_PIPELINE_20260831"
assert (PROJECT_ROOT / "src" / "dam_pipeline").is_dir(), PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(sys.version)
print(sys.executable)
assert sys.version_info[:3] == (3, 12, 13)
assert Path(sys.executable).resolve() == Path.home() / "miniconda3" / "envs" / "pfe" / "python.exe"
print("KERNEL_GATE = PASS")
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Single user configuration cell
PAGE1_PDF_ROOT = Path.home() / "Downloads" / "dam_pdfs_split" / "raw_data" / "1"
PAGE2_PDF_ROOT = Path.home() / "Downloads" / "dam_pdfs_split" / "raw_data" / "2"
OUTPUT_BASE = PROJECT_ROOT / "outputs" / "runs"
RUN_ID = "RUN_20260831_FULL_01"
PAGE1_WORKERS = 8
PAGE2_WORKERS = 12
RUN_FULL_CORPUS = True
WRITE_SQLITE = True
WRITE_CSV = True
GENERATE_FIGURES = True
RUN_DIR = OUTPUT_BASE / RUN_ID
print("RUN_DIR =", RUN_DIR)

In [ ]:
from dam_pipeline.corpus import sha256_file
hashes = pd.read_csv(PROJECT_ROOT / "investigation" / "ACCEPTED_SOURCE_HASHES.csv")
current = []
for row in hashes.itertuples():
    path = PROJECT_ROOT / row.destination_path
    current.append({"destination_path": row.destination_path, "expected": row.sha256, "actual": sha256_file(path), "status": "PASS" if sha256_file(path) == row.sha256 else "FAIL"})
hash_check = pd.DataFrame(current)
display(hash_check)
assert (hash_check.status == "PASS").all()
assert sha256_file(PROJECT_ROOT / "config" / "page1" / "PAGE1_GOVERNED_ALIAS_REGISTER.csv") == "aff3b38b3bf42b3b3d059dccd33857affe4ddc173ccff3b5aa00f851c33324dc"
print("FROZEN_SOURCE_HASHES = PASS")

In [ ]:
from dam_pipeline.corpus import inventory, compare_accepted
expected_counts = (3981, 3983) if RUN_FULL_CORPUS else (4, 4)
root1 = PAGE1_PDF_ROOT if RUN_FULL_CORPUS else PROJECT_ROOT / "fixtures" / "page1"
root2 = PAGE2_PDF_ROOT if RUN_FULL_CORPUS else PROJECT_ROOT / "fixtures" / "page2"
page1_inventory = inventory(root1, 1, expected_counts[0])
page2_inventory = inventory(root2, 2, expected_counts[1])
source_inventory = pd.concat([page1_inventory, page2_inventory], ignore_index=True)
accepted_check = compare_accepted(source_inventory, PROJECT_ROOT / "references" / "COMBINED_SOURCE_INVENTORY.csv")
assert (accepted_check.status == "PASS").all()
display(source_inventory.groupby("page").agg(files=("source_file","size"), bytes=("source_bytes","sum")))
print("FIRST/LAST PAGE1 =", page1_inventory.source_file.iloc[[0,-1]].tolist())
print("FIRST/LAST PAGE2 =", page2_inventory.source_file.iloc[[0,-1]].tolist())
print("CORPUS_GATE = PASS")

## Extraction

The new runner inventories and hashes direct PDF children, verifies them against the accepted Run-5 inventory, then invokes the frozen accepted APIs. Each page writes observations, cell lineage, source manifests, checksum/validation evidence, failures, governed protected source states, and its own clean-room database before the new merge layer begins.

In [ ]:
from dam_pipeline.run_extraction_pipeline import run_pipeline
assert RUN_FULL_CORPUS, "The production notebook must execute the full corpus."
receipt_path = RUN_DIR / "pipeline_receipt.json"
if receipt_path.is_file():
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    assert receipt["status"] == "COMPLETE"
    assert receipt["source_counts"] == {"page1": 3981, "page2": 3983}
    print("REUSING_COMPLETED_FULL_RUN_RECEIPT =", receipt_path)
else:
    receipt = run_pipeline(
        root1, root2, RUN_DIR, run_id=RUN_ID, project_root=PROJECT_ROOT,
        page1_workers=PAGE1_WORKERS, page2_workers=PAGE2_WORKERS,
        expected_page1_count=3981, expected_page2_count=3983,
        write_sqlite=WRITE_SQLITE, write_csv=WRITE_CSV, generate_figures=GENERATE_FIGURES,
    )
receipt_display = json.loads(json.dumps(receipt, default=str))
page2_display = receipt_display.get("page2_summary", {})
if "unresolved_count" in page2_display:
    page2_display["preserved_state_reporting_union_count"] = page2_display.pop("unresolved_count")
print(json.dumps(receipt_display, indent=2, default=str))

In [ ]:
page1 = pd.read_csv(RUN_DIR / "page1" / "observations.csv.gz", low_memory=False)
page1_summary = pd.read_csv(RUN_DIR / "page1" / "run_summary.csv")
display(page1_summary.T)
display(page1.head())
assert len(page1) == receipt["page1_observation_count"]

In [ ]:
receipt = json.loads((RUN_DIR / "pipeline_receipt.json").read_text(encoding="utf-8"))
page2 = pd.read_csv(RUN_DIR / "page2" / "observations.csv.gz", low_memory=False)
page2_summary = pd.read_csv(RUN_DIR / "page2" / "run_summary.csv")
page2_states = pd.read_csv(RUN_DIR / "page2" / "unresolved_values.csv", low_memory=False)
rs_states = page2_states[page2_states["canonical_semantic"].eq("p2_rs_unresolved_prohibited")]
nonnumeric_expressions = page2_states[page2_states["disposition"].eq("SOURCE_EXPRESSION_PRESERVED_NO_INFERENCE")]
quality_status = pd.DataFrame([
    {"quality_category":"Page-2 extraction failures","rows":int(page2_summary.loc[0,"failure_count"]),"actionable":True},
    {"quality_category":"Page-2 actionable parser ambiguities","rows":int(page2_summary.loc[0,"ambiguous_row_count"]),"actionable":True},
    {"quality_category":"Page-2 protected R.S. states","rows":len(rs_states),"actionable":False},
    {"quality_category":"Numeric parsed but deliberately uninterpreted","rows":int(rs_states["parse_status"].eq("NUMERIC_PARSED_RS_UNRESOLVED_PROHIBITED").sum()),"actionable":False},
    {"quality_category":"Source blanks preserved as missing","rows":int(rs_states["parse_status"].eq("SOURCE_BLANK").sum()),"actionable":False},
    {"quality_category":"Page-2 preserved nonnumeric source expressions","rows":len(nonnumeric_expressions),"actionable":False},
])
assert quality_status["rows"].tolist() == [0,0,129472,81445,48027,16223]
assert len(page2_states) == 145695 == len(rs_states) + len(nonnumeric_expressions)
assert page2_summary.loc[0,"scientific_identity_hash"] == "8b48836811dcfe80fe12db79295d360f1980e54ab4e51f69f559ab87ef55d361"
page2_summary_display = page2_summary.rename(columns={"unresolved_count":"preserved_state_reporting_union_count"})
display(page2_summary_display.T)
display(quality_status)
display(Markdown("**Scientific interpretation:** 145,695 is the union of protected R.S. states and preserved nonnumeric expressions. It is not a count of failed observations. Page‑2 has zero extraction failures and zero actionable parser ambiguities."))
display(page2.head())
assert len(page2) == receipt["page2_observation_count"]

## Merge contract

Keys are `canonical_dam_id + bulletin_date`; only Page-2's structural key name `dam_id` is normalized. Keys must be unique. Every non-key field is page-prefixed. A validated one-to-one outer merge preserves `both`, `left_only`, and `right_only`, sorted by date then dam. Scientific measurements are never renamed, combined, averaged, inferred, interpolated, or zero-filled.

In [ ]:
from dam_pipeline.merge import merge_pages
merged = merge_pages(page1, page2)
assert len(merged) == receipt["merged_row_count"]
assert not merged.duplicated(["canonical_dam_id","bulletin_date"]).any()
display(merged.page_presence.value_counts(dropna=False).rename("rows"))
print("MERGED_SHAPE =", merged.shape)

In [ ]:
database_path = RUN_DIR / "database" / "dam_data.db"
with sqlite3.connect(f"file:{database_path.as_posix()}?mode=ro", uri=True) as connection:
    integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
    foreign_keys = connection.execute("PRAGMA foreign_key_check").fetchall()
    table_counts = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection)
assert integrity == "ok" and not foreign_keys
print("DATABASE_INTEGRITY =", integrity)
display(table_counts)

In [ ]:
plain_path = RUN_DIR / "exports" / "merged_dam_daily.csv"
gzip_path = RUN_DIR / "exports" / "merged_dam_daily.csv.gz"
plain = pd.read_csv(plain_path, low_memory=False)
compressed = pd.read_csv(gzip_path, low_memory=False)
pd.testing.assert_frame_equal(plain, compressed, check_dtype=False)
print("MERGED_CSV_SHA256 =", sha256_file(plain_path))
print("ROWS/COLUMNS/DAMS =", len(plain), len(plain.columns), plain.canonical_dam_id.nunique())
print("DATE_RANGE =", plain.bulletin_date.min(), plain.bulletin_date.max())

## Investigation methodology

Coverage, missingness, lineage states, checksum dispositions, actionable failures, non-actionable protected source states, and three cross-page comparisons are computed only from this fresh run. Figure titles record sample size and state that missing pairs are excluded. Cross-page plots preserve both source values and measure differences without asserting equivalence.

In [ ]:
from dam_pipeline.corpus import sha256_file
investigation_files = sorted((RUN_DIR / "investigation").glob("*.csv"))
assert len(investigation_files) == 14
investigation_index = pd.DataFrame({"artifact":[p.name for p in investigation_files],"rows":[len(pd.read_csv(p)) for p in investigation_files],"sha256":[sha256_file(p) for p in investigation_files]})
display(investigation_index)

In [ ]:
from PIL import Image
figure_files = sorted((RUN_DIR / "figures").glob("*.png"))
assert len(figure_files) == 12
assert "08_quality_status.png" in {p.name for p in figure_files}
assert "08_failures_and_unresolved.png" not in {p.name for p in figure_files}
for path in figure_files:
    if path.name == "08_quality_status.png":
        interpretation = (
            "This quality-status figure separates actionable outcomes (zero extraction failures; "
            "zero actionable parser ambiguities) from governed non-actionable source preservation. "
            "The 145,695 reporting union is 129,472 protected R.S. states plus 16,223 preserved "
            "nonnumeric source expressions; it is not a count of failed observations."
        )
    else:
        interpretation = (
            "Missing values are excluded from plotted comparisons; see the title for n. "
            "This visual describes the fresh run and should be interpreted alongside its machine-readable CSV."
        )
    display(Markdown(f"### {path.stem.replace('_',' ').title()}\n\n{interpretation}"))
    display(Image.open(path))

In [ ]:
display(merged)
print(f"Displayed all {len(merged):,} merged observations × {len(merged.columns)} columns.")

In [ ]:
final_receipt = json.loads((RUN_DIR / "pipeline_receipt.json").read_text(encoding="utf-8"))
assert final_receipt["source_counts"] == {"page1":3981,"page2":3983}
assert final_receipt["database"]["integrity_check"] == "ok"
assert final_receipt["accepted_inventory_match"] is True
final_receipt_display = json.loads(json.dumps(final_receipt))
page2_final = final_receipt_display.get("page2_summary", {})
if "unresolved_count" in page2_final:
    page2_final["preserved_state_reporting_union_count"] = page2_final.pop("unresolved_count")
final_receipt_display["page2_quality_status"] = {
    "extraction_failures": 0,
    "actionable_parser_ambiguities": 0,
    "protected_rs_states": 129472,
    "numeric_protected_rs": 81445,
    "protected_rs_source_blanks": 48027,
    "preserved_nonnumeric_source_expressions": 16223,
    "reporting_union_note": "145,695 is a reporting union, not failed observations",
}
final_receipt_display["production_notebook_complete"] = True
display(pd.json_normalize(final_receipt_display))
print("PRODUCTION_EXECUTION_COMPLETE = TRUE")